# Table 2 Replication
This notebook replicates the Global and Japan sections of Table 2 in Fama and French (2012).

Panel A reports the mean and standard deviation of monthly excess returns for 25 portfolios formed on size and book-to-market.

Panel B reports the same statistics for 25 portfolios formed on size and previous returns.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
DATA_DIR = Path("cleaned_data")

In [ ]:
developed_factors = pd.read_csv(
    DATA_DIR / "developed_3_factors.csv",
    parse_dates=["date"]
)

In [ ]:
japan_factors = pd.read_csv(
    DATA_DIR / "japan_3_factors.csv",
    parse_dates=["date"]
)

In [ ]:
developed_size_bm = pd.read_csv(
    DATA_DIR / "developed_25_size_bm.csv",
    parse_dates=["date"]
)

In [ ]:
japan_size_bm = pd.read_csv(
    DATA_DIR / "japan_25_size_bm.csv",
    parse_dates=["date"]
)

In [ ]:
developed_size_momentum = pd.read_csv(
    DATA_DIR / "developed_25_size_momentum.csv",
    parse_dates=["date"]
)

In [ ]:
japan_size_momentum = pd.read_csv(
    DATA_DIR / "japan_25_size_momentum.csv",
    parse_dates=["date"]
)

In [ ]:
print("Developed size and B/M:", len(developed_size_bm))
print("Japan size and B/M:", len(japan_size_bm))
print("Developed size and momentum:", len(developed_size_momentum))
print("Japan size and momentum:", len(japan_size_momentum))

## Matching portfolio returns with the risk-free rate

Before calculating excess returns, match the portfolio data with the factor data using `date`.

Use the risk-free rate from the corresponding market:

- Use Developed `RF` for both Developed portfolio datasets.
- Use Japanese `RF` for both Japanese portfolio datasets.

For each month, subtract the same market-specific `RF` from all 25 portfolio returns.

Subtract only the `RF` column. Do not subtract `Mkt-RF`.

The portfolio returns and `RF` are already expressed as percentages. Do not divide them by 100 or annualize them.

## Task 1: Calculate Excess Returns

Table 2 reports excess portfolio returns.

For every month, calculate:

Portfolio excess return = portfolio return − risk-free return

The risk-free rate, `RF`, is available in the factor datasets loaded above.

Calculate excess returns for these four combinations:

1. Developed size and book-to-market portfolios
2. Japan size and book-to-market portfolios
3. Developed size and momentum portfolios
4. Japan size and momentum portfolios

Keep the date and all 25 excess-return columns.

In [1]:
# Write your code here

import pandas as pd

# Load factor data for RF retrieval
dev_3f = pd.read_csv("cleaned_data/developed_3_factors.csv", parse_dates=["date"])
jp_3f = pd.read_csv("cleaned_data/japan_3_factors.csv", parse_dates=["date"])

# Load portfolio datasets
dev_bm = pd.read_csv(
    "cleaned_data/developed_25_size_bm.csv", parse_dates=["date"]
)
dev_mom = pd.read_csv(
    "cleaned_data/developed_25_size_momentum.csv", parse_dates=["date"]
)
jp_bm = pd.read_csv(
    "cleaned_data/japan_25_size_bm.csv", parse_dates=["date"]
)
jp_mom = pd.read_csv(
    "cleaned_data/japan_25_size_momentum.csv", parse_dates=["date"]
)


# Function to calculate excess returns by subtracting RF
def compute_excess(df_port, df_factor):
  merged = pd.merge(df_port, df_factor[["date", "RF"]], on="date", how="inner")
  portfolio_cols = [
      col for col in df_port.columns if col != "date"
  ]  # all 25 portfolios
  excess_df = merged[["date"]].copy()
  for col in portfolio_cols:
    excess_df[col] = merged[col] - merged["RF"]
  return excess_df


# Calculate excess returns for all 4 combinations
dev_bm_excess = compute_excess(dev_bm, dev_3f)
jp_bm_excess = compute_excess(jp_bm, jp_3f)
dev_mom_excess = compute_excess(dev_mom, dev_3f)
jp_mom_excess = compute_excess(jp_mom, jp_3f)

print("Excess returns calculated successfully for all 4 sets!")

Excess returns calculated successfully for all 4 sets!


## Constructing the Table 2 results

Calculate the statistics separately for every portfolio.

For each portfolio:

1. Use its 245 monthly excess returns.
2. Calculate one arithmetic mean.
3. Calculate one sample standard deviation.

This produces 25 means and 25 standard deviations for each dataset.

Do not calculate the mean across the 25 portfolios. The averaging is performed across the 245 months for each portfolio separately.

Arrange the results in the same order as the portfolio columns in the data:

- The first five portfolio columns form the `Small` row.
- The next five form size row `2`.
- The next five form size row `3`.
- The next five form size row `4`.
- The final five form the `Big` row.

## Task 2: Report the Portfolio Statistics

For each of the four combinations, calculate:

1. Mean monthly excess return
2. Standard deviation of monthly excess return

Arrange each result as a 5 × 5 table (refer to the paper to understand the structure).

For the size and book-to-market portfolios:

- Rows should move according to size from Small to Big.
- Columns should move from Low book-to-market to High book-to-market.

For the size and momentum portfolios:

- Rows should move according to size from Small to Big.
- Columns should move according to the momentum factor from Losers to Winners.

Round the results to two decimal places.

In [2]:
# Write your code here
import numpy as np


# Helper function to shape 25 columns into a 5x5 matrix
def create_5x5_table(excess_df, stat_type="mean"):
  portfolio_cols = [col for col in excess_df.columns if col != "date"]

  if stat_type == "mean":
    stats = excess_df[portfolio_cols].mean()
  else:
    stats = excess_df[portfolio_cols].std(ddof=1)

  # Reshape the 25 values into a 5x5 array (5 size rows x 5 characteristic columns)
  matrix = stats.values.reshape(5, 5)

  # Create a clean DataFrame
  row_labels = ["Small", "2", "3", "4", "Big"]
  col_labels = ["Low", "2", "3", "4", "High"]
  return pd.DataFrame(matrix, index=row_labels, columns=col_labels).round(2)


print("--- 1. Developed Size and B/M ---")
print("Mean Excess Returns:")
display(create_5x5_table(dev_bm_excess, "mean"))
print("Standard Deviations:")
display(create_5x5_table(dev_bm_excess, "std"))

print("\n--- 2. Japan Size and B/M ---")
print("Mean Excess Returns:")
display(create_5x5_table(jp_bm_excess, "mean"))
print("Standard Deviations:")
display(create_5x5_table(jp_bm_excess, "std"))

print("\n--- 3. Developed Size and Momentum ---")
print("Mean Excess Returns:")
display(create_5x5_table(dev_mom_excess, "mean"))
print("Standard Deviations:")
display(create_5x5_table(dev_mom_excess, "std"))

print("\n--- 4. Japan Size and Momentum ---")
print("Mean Excess Returns:")
display(create_5x5_table(jp_mom_excess, "mean"))
print("Standard Deviations:")
display(create_5x5_table(jp_mom_excess, "std"))


--- 1. Developed Size and B/M ---
Mean Excess Returns:


,Low,2,3,4,High
Small,0.05,0.48,0.77,0.78,1.12
2,0.08,0.42,0.53,0.67,0.80
3,0.19,0.39,0.52,0.58,0.76
4,0.40,0.43,0.47,0.62,0.68
Big,0.28,0.37,0.48,0.52,0.54


Standard Deviations:


,Low,2,3,4,High
Small,6.07,5.59,5.31,4.60,4.38
2,6.01,5.32,4.65,4.39,4.49
3,5.88,5.26,4.69,4.46,4.60
4,5.78,4.64,4.54,4.41,4.73
Big,4.65,4.30,4.45,4.48,5.25



--- 2. Japan Size and B/M ---
Mean Excess Returns:


...

,Low,2,3,4,High
Small,6.41,4.35,3.91,4.11,5.43
2,6.73,4.67,4.19,4.20,5.59
3,6.73,4.87,4.28,4.22,5.54
4,6.70,4.82,4.21,4.16,5.40
Big,6.31,4.64,4.09,4.16,5.31



--- 4. Japan Size and Momentum ---
Mean Excess Returns:


,Low,2,3,4,High
Small,0.15,0.30,0.13,0.28,-0.01
2,-0.14,-0.05,0.01,-0.03,-0.10
3,-0.21,-0.22,-0.13,-0.03,-0.05
4,-0.11,-0.11,-0.14,-0.17,-0.01
Big,-0.08,-0.30,-0.28,-0.10,-0.07


Standard Deviations:


,Low,2,3,4,High
Small,8.84,7.18,6.62,6.58,8.02
2,8.72,7.07,6.64,6.70,7.45
3,8.09,6.87,6.07,6.19,6.98
4,8.02,6.54,6.07,5.91,6.78
Big,8.36,6.54,6.15,5.88,6.86


## Task 3: Compare and Interpret the Results

Compare your size and book-to-market results with the Global and Japan sections of Table 2, Panel A.

Compare your size and momentum results with the Global and Japan sections of Table 2, Panel B.

Identify the main return patterns and differences that you would report. Interpret what the results show about value and momentum across company sizes and across the two markets.

Your values may differ slightly because the Kenneth French database has been updated since the paper was published.

Note your findings in your report.